<div dir="rtl">
<h1>مشتق خودکار؛ وزن هنوز ثابت است</h1>
<p>درس 21 از 76 · مشتق خودکار چه چیزی را ثبت می‌کند؟ · <code dir="ltr">17-autograd</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">📖 بازگشت به همین درس</a></p>
<p>ثبت Graph، محاسبهٔ Gradient و تغییر وزن را با سه شاهد جدا بررسی کنید.</p><p>پیش‌نیاز: 12-chain و کار با Tensorهای اعشاری؛ Autograd در درس جاری.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>پس از backward برای (2w-5)² در w=1، مقدار w چیست و w.grad چیست؟ اجرای تازهٔ دوباره بدون پاک‌کردن Gradient چه می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
torch.set_num_threads(1)
print("Reference: w=1, input=2, target=5")

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع inspect_gradient(Value) Tensor تک‌عنصری تازه با requires_grad=True بسازد، Loss=(2w-5)² را backward کند و tupleِ عددی (Loss, Gradient, unchanged_weight) برگرداند. هیچ update انجام ندهید.</p>
</div>

In [ ]:
def inspect_gradient(value):
    # TODO: return three Python numbers after backward
    return None

In [ ]:
def test_exercise():
    result = inspect_gradient(1.)
    if result is None:
        return False
    assert result == (9., -12., 1.)
    assert inspect_gradient(2.) == (1., -4., 2.)
    assert inspect_gradient(2.5) == (0., 0., 2.5)
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط پاک‌کردن Gradient بین دو اجرای تازه را روشن و خاموش کنید؛ وزن در هر دو آزمایش ثابت بماند.</p>
</div>

In [ ]:
for clear in [False, True]:
    w = torch.tensor(1., requires_grad=True)
    values = []
    for _ in range(2):
        if clear:
            w.grad = None
        ((2*w-5)**2).backward()
        values.append(w.grad.item())
    print("clear:", clear, "gradients:", values, "weight:", w.item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>item برای گزارش است؛ عدد Python مسیر مشتق ندارد. تابع connected_loss باید Loss Tensor را برگرداند تا فراخواننده بتواند backward کند.</p>
</div>

In [ ]:
w = torch.tensor(1., requires_grad=True)
wrong = ((2*w-5)**2).item()
try:
    wrong.backward()
except AttributeError as error:
    print("Expected graph loss:", error)
else:
    raise AssertionError("A Python number has no backward")

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def connected_loss(w):
    # TODO: keep the tensor connected to w
    return None

In [ ]:
def test_repair():
    result = connected_loss(w)
    if result is None:
        return False
    assert isinstance(result, torch.Tensor) and result.requires_grad
    w.grad = None
    result.backward()
    assert w.grad.item() == -12
    fresh = torch.tensor(2., requires_grad=True)
    connected_loss(fresh).backward()
    assert fresh.grad.item() == -4
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>mini_gpt/train.py مقدار item را برای گزارش می‌گیرد، اما backward را روی خود Tensor Loss اجرا می‌کند. همان جدایی ساده، در مدل بزرگ هم ضروری است.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر Gradient درست باشد ولی وزن عوض نشود، آیا مشکل الزاماً در Autograd است؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-03/17-autograd.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/17-autograd.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>